# LS估计、MMSE估计、低复杂度方法

**LS and MMSE Channel Estimation, Low-Complexity Methods**

---

本notebook介绍信道估计的两种基本方法：最小二乘（LS）估计和最小均方误差（MMSE）估计，以及在贝叶斯框架下的信道估计理论。我们还将探讨低复杂度方法（SVD-MMSE、时域平滑），并比较不同SNR下的估计性能。

## 1. 学习目标 (Objectives)

通过本notebook，您将：

- **理解LS估计的原理**：掌握最小二乘准则在信道估计中的应用
- **掌握MMSE估计**：理解均方误差最小化准则和线性MMSE实现
- **认识贝叶斯框架**：了解先验知识如何提升估计性能
- **掌握低复杂度方法**：学习SVD-MMSE和时域平滑等实用技术
- **比较不同SNR下的性能**：通过仿真观察LS和MMSE在不同信噪比下的表现
- **为OTFS信道估计打下基础**：理解延迟-多普勒域的信道估计方法

## 2. 信道估计问题建模 (Problem Formulation)

### 2.1 系统模型

考虑OFDM系统中的信道估计问题。假设有$N$个子载波，信道包含$L$个路径。

**发送端**：
- QAM符号向量 $\mathbf{X} = [X_0, X_1, ..., X_{N-1}]^T$
- 经过IFFT调制后发送

**信道**：
- 时域冲激响应 $h[n]$，长度$L$（假设$L \ll N$）
- 频域信道响应 $\mathbf{H} = \mathbf{F}_L \mathbf{h}$，其中 $\mathbf{F}_L$ 是FFT矩阵的前$L$列

**接收端**：
$$Y[k] = X[k] \cdot H[k] + W[k], \quad k = 0, 1, ..., N-1$$

其中 $W[k]$ 是复高斯噪声，$W[k] \sim \mathcal{CN}(0, \sigma_w^2)$。

### 2.2 估计问题的矩阵形式

将OFDM系统模型写成矩阵形式：

$$\mathbf{Y} = \mathbf{X} \mathbf{H} + \mathbf{W}$$

其中：
- $\mathbf{Y} = [Y_0, Y_1, ..., Y_{N-1}]^T$ 是接收信号向量
- $\mathbf{X} = \text{diag}(X_0, X_1, ..., X_{N-1})$ 是发送符号构成的对角矩阵
- $\mathbf{H} = [H_0, H_1, ..., H_{N-1}]^T$ 是频域信道响应向量
- $\mathbf{W} = [W_0, W_1, ..., W_{N-1}]^T$ 是噪声向量

**信道估计的目标**：从 $\mathbf{Y}$ 和已知（导频）符号 $\mathbf{X}$ 估计信道 $\mathbf{H}$。

## 3. 最小二乘（LS）估计 (Least Squares Estimation)

### 3.1 LS估计原理

最小二乘（LS）估计的核心思想是：**找到使得残差平方和最小的估计值**。

LS准则：
$$\hat{\mathbf{H}}_{LS} = \arg\min_{\mathbf{H}} \| \mathbf{Y} - \mathbf{X}\mathbf{H} \|^2$$

对 $\mathbf{H}$ 求导并令导数为零：
$$\frac{\partial}{\partial \mathbf{H}^*} \left( \mathbf{Y} - \mathbf{X}\mathbf{H} \right)^H \left( \mathbf{Y} - \mathbf{X}\mathbf{H} \right) = 0$$

求解得到LS估计：
$$\boxed{\hat{\mathbf{H}}_{LS} = \mathbf{X}^{-1} \mathbf{Y}}$$

由于 $\mathbf{X}$ 是对角矩阵，上式简化为：
$$\hat{H}_{LS}[k] = \frac{Y[k]}{X[k]}, \quad k = 0, 1, ..., N-1$$

### 3.2 LS估计的统计特性

**噪声放大效应**：

将 $Y[k] = X[k]H[k] + W[k]$ 代入：

$$\hat{H}_{LS}[k] = H[k] + \frac{W[k]}{X[k]}$$

因此：
- **估计是无偏的**：$E\{\hat{H}_{LS}[k]\} = H[k]$
- **估计误差**：$\epsilon[k] = \hat{H}_{LS}[k] - H[k] = W[k]/X[k]$
- **噪声方差**：$\text{Var}\left(\hat{H}_{LS}[k]\right) = \frac{\sigma_w^2}{|X[k]|^2}$

**问题**：当 $|X[k]|^2$ 很小时，噪声被放大！这是LS估计的主要缺点。

### 3.3 LS估计的均方误差（MSE）

LS估计的MSE为：

$$\text{MSE}_{LS} = E\left[|\hat{H}_{LS}[k] - H[k]|^2\right] = \frac{\sigma_w^2}{|X[k]|^2}$$

如果所有导频符号功率相同（$|X[k]|^2 = 1$）：

$$\text{MSE}_{LS} = \sigma_w^2$$

注意：LS估计的MSE与信噪比成反比，无法利用信道的任何结构信息。

## 4. MMSE（最小均方误差）估计 (Minimum Mean Square Error Estimation)

### 4.1 MMSE估计原理

MMSE估计的核心思想是：**利用信道的统计特性（先验知识），最小化均方误差**。

MMSE准则：
$$\hat{\mathbf{H}}_{MMSE} = \arg\min_{\hat{\mathbf{H}}} E\left[\|\mathbf{H} - \hat{\mathbf{H}}\|^2\right]$$

对于线性模型 $Y = XH + W$，如果 $H$ 和 $W$ 独立且服从高斯分布，则MMSE估计是**线性的**：

$$\boxed{\hat{\mathbf{H}}_{MMSE} = \mathbf{R}_{HH} \left(\mathbf{R}_{HH} + \sigma_w^2 (\mathbf{X}^H\mathbf{X})^{-1}\right)^{-1} \hat{\mathbf{H}}_{LS}}}$

### 4.2 逐子载波的MMSE估计

假设各子载波信道独立，且 $|X[k]|^2 = 1$，MMSE估计简化为：

$$\hat{H}_{MMSE}[k] = \frac{R_{HH}[k,k]}{R_{HH}[k,k] + \sigma_w^2} \cdot \hat{H}_{LS}[k]$$

其中 $R_{HH} = E\{\mathbf{H}\mathbf{H}^H\}$ 是信道自相关矩阵。

**物理意义**：
- 当 $\sigma_w^2 \to 0$（高SNR）：$\hat{H}_{MMSE}[k] \approx \hat{H}_{LS}[k]$，MMSE退化为LS
- 当 $\sigma_w^2 \to \infty$（低SNR）：$\hat{H}_{MMSE}[k] \approx 0$，完全依赖先验均值
- 当 $\sigma_w^2$ 中等时：MMSE是LS估计和先验均值的加权平均

### 4.3 MMSE估计 vs LS估计

| 特性 | LS估计 | MMSE估计 |
|------|--------|----------|
| **计算复杂度** | 极低 $O(N)$ | 中等 $O(N^2)$ 或更高 |
| **利用先验知识** | 否 | 是（信道统计特性） |
| **估计偏差** | 无偏 | 可能是有偏的 |
| **MSE性能** | 较差 | 最优（在MSE准则下） |
| **噪声放大** | 严重 | 被抑制 |
| **需要信息** | 仅导频符号 | 导频 + 信道统计特性 |

## 5. 贝叶斯框架下的信道估计 (Bayesian Channel Estimation)

### 5.1 贝叶斯估计理论

贝叶斯估计将待估计参数视为随机变量，利用先验分布建模其统计特性。

**先验分布**：$H[k] \sim \mathcal{CN}(0, \sigma_H^2)$，即信道服从复高斯分布

**后验分布**：
$$p(H[k]|Y[k]) = \frac{p(Y[k]|H[k]) \cdot p(H[k])}{p(Y[k])}$$

**贝叶斯MSE**：
$$\text{BMSE} = E_{H[k], Y[k]}\left[|H[k] - \hat{H}[k]|^2\right]$$

### 5.2 最大后验（MAP）估计

MAP估计选择使后验概率最大的估计值：

$$\hat{H}_{MAP} = \arg\max_{H} p(H|Y) = \arg\max_{H} p(Y|H) \cdot p(H)$$

假设 $p(Y|H) \sim \mathcal{CN}(XH, \sigma_w^2)$ 和 $p(H) \sim \mathcal{CN}(0, \sigma_H^2)$：

$$\hat{H}_{MAP}[k] = \frac{\sigma_H^2}{\sigma_H^2 + \sigma_w^2/|X[k]|^2} \cdot \hat{H}_{LS}[k]$$

有趣的是：**当信道和噪声都服从高斯分布时，MAP估计与MMSE估计等价！**

### 5.3 不同先验分布的影响

实际信道往往不是高斯分布，而是**稀疏的**（只有少数几条路径）。

**稀疏先验**：
- $p(H) = \sum_{i=0}^{L} w_i \cdot \delta(H - H_i)$，其中大部分 $H_i \approx 0$
- 利用压缩感知技术进行稀疏恢复

**Laplace先验**（稀疏诱导先验）：
$$p(H) \propto \exp\left(-\sqrt{2}\sum_k |H[k]|\right)$$

稀疏MMSE估计可以在低SNR情况下保持信道稀疏性，提升估计精度。

## 6. 低复杂度方法 (Low-Complexity Methods)

### 6.1 SVD-MMSE

MMSE估计需要计算 $N \times N$ 矩阵的逆，复杂度为 $O(N^3)$。

**SVD-MMSE** 利用信道自相关矩阵的低秩特性：

1. 对 $R_{HH}$ 进行SVD分解：
   $$R_{HH} = U \Sigma V^H$$
   
2. 只保留前$r$个主奇异值（$r \ll N$）：
   $$R_{HH} \approx U_r \Sigma_r V_r^H$$
   
3. MMSE估计变为：
   $$\hat{H}_{SVD-MMSE} = U_r \Sigma_r (\Sigma_r + \sigma_w^2 I)^{-1} V_r^H \hat{H}_{LS}$$

### 6.2 SVD-MMSE的复杂度分析

| 方法 | 复杂度 | 存储需求 |
|------|--------|----------|
| 精确MMSE | $O(N^3)$ | $O(N^2)$ |
| SVD-MMSE | $O(rN^2)$ | $O(rN)$ |
| 对角化MMSE | $O(N^2)$ | $O(N)$ |

其中 $r$ 是保留的秩，通常 $r \ll N$（例如 $r = 8$ 当 $N = 64$）。

### 6.3 时域平滑（Temporal Smoothing）

对于时变信道，可以利用**相邻OFDM符号之间的相关性**进行平滑：

$$\bar{H}[k,l] = \alpha \cdot \hat{H}_{LS}[k,l] + (1-\alpha) \cdot \bar{H}[k,l-1]$$

其中：
- $\alpha$ 是平滑系数（$0 < \alpha \le 1$）
- $l$ 是OFDM符号索引
- $\bar{H}[k,l]$ 是平滑后的信道估计

**选择$\alpha$**：
- $\alpha = 1$：无平滑，等于LS
- $\alpha = 0$：完全依赖历史估计
- $\alpha = \frac{1}{1+\ SNR}$：自适应选择

### 6.4 频域插值 + 时域平滑

实际系统中，导频是**梳状**（comb-type）分布的，即每隔若干子载波放置一个导频。

**两步估计法**：

1. **导频位置LS估计**：对导频子载波进行LS估计
   $$\hat{H}_{p}[k] = \frac{Y_p[k]}{X_p[k]}, \quad k \in \{p_1, p_2, ..., p_N\}$$
   
2. **频域插值**：利用导频处的估计值，插值得到所有子载波的信道估计
   - 线性插值
   - 样条插值
   - 基于DFT的插值（利用信道时域稀疏性）
   
3. **时域平滑**：对相邻符号进行平滑

## 7. 代码演示：LS和MMSE估计 (Implementation)

下面我们实现LS估计和MMSE估计，并比较它们在不同SNR下的性能。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import linalg
%matplotlib inline

# 设置中文字体
plt.rcParams['font.sans-serif'] = ['SimHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

print("信道估计演示包加载成功")

### 7.1 系统参数设置

In [ ]:
# OFDM系统参数
N = 64          # 子载波数量
L = 4           # 信道路径数
cp_len = 16     # 循环前缀长度
num_symbols = 100  # OFDM符号数量（用于时域平滑）

# 信道模型参数
channel_delay_spread = 3  # 信道延迟扩展（采样点数）
channel_doppler = 0       # 信道多普勒扩展（归一化）

# 导频参数（梳状导频）
pilot_interval = 8   # 导频间隔
pilot_indices = np.arange(0, N, pilot_interval)  # 导频位置
num_pilots = len(pilot_indices)

print(f"子载波数 N = {N}")
print(f"信道路径数 L = {L}")
print(f"导频数量 = {num_pilots} (每隔{pilot_interval}个子载波一个导频)")
print(f"导频位置索引 = {list(pilot_indices)}")

### 7.2 生成信道和导频

In [ ]:
def generate_channel(num_paths, max_delay, snr_db):
    """
    生成多径信道
    
    参数:
        num_paths: 路径数量
        max_delay: 最大延迟（采样点）
        snr_db: 信噪比（dB）
    
    返回:
        h_time: 时域信道冲激响应
        h_freq: 频域信道响应
        snr_linear: 线性信噪比
    """
    np.random.seed(42)
    
    # 时域冲激响应（稀疏：只有num_paths个非零点）
    h_time = np.zeros(max_delay + 5, dtype=complex)
    
    # 生成路径：延迟和幅度
    delays = np.sort(np.random.choice(np.arange(max_delay + 1), size=num_paths, replace=False))
    amplitudes = (np.random.randn(num_paths) + 1j * np.random.randn(num_paths)) / np.sqrt(2)
    
    for i, (d, a) in enumerate(zip(delays, amplitudes)):
        h_time[d] = a
    
    # 频域信道响应（FFT）
    h_freq = np.fft.fft(h_time, n=N)
    
    # 计算SNR
    signal_power = np.mean(np.abs(h_freq)**2)
    snr_linear = 10**(snr_db / 10)
    noise_variance = signal_power / snr_linear
    
    return h_time, h_freq, snr_linear, noise_variance


def generate_pilot_symbols(pilot_indices, qam_order=4):
    """
    生成导频符号（QPSK）
    
    参数:
        pilot_indices: 导频位置索引
        qam_order: QAM阶数
    
    返回:
        pilot_symbols: 导频符号
    """
    # QPSK星座点
    qpsk = np.array([1+0j, 0+1j, -1+0j, 0-1j]) / np.sqrt(2)
    np.random.seed(100)
    pilot_symbols = np.random.choice(qpsk, size=len(pilot_indices))
    return pilot_symbols


# 测试：生成信道
h_time, h_freq, snr_lin, noise_var = generate_channel(L, channel_delay_spread, snr_db=20)
pilot_symbols = generate_pilot_symbols(pilot_indices)

print("信道参数：")
print(f"  时域冲激响应长度: {len(h_time)}")
print(f"  时域非零点数量: {np.sum(np.abs(h_time) > 0)}")
print(f"  信噪比: {20} dB (线性: {snr_lin:.2f})")
print(f"  噪声方差: {noise_var:.6f}")

### 7.3 LS估计实现

In [ ]:
def ls_estimation(rx_pilot_signals, pilot_symbols):
    """
    最小二乘（LS）信道估计
    
    参数:
        rx_pilot_signals: 接收到的导频信号 (num_pilots,)
        pilot_symbols: 发送的导频符号 (num_pilots,)
    
    返回:
        h_ls_pilot: 导频位置的LS信道估计
    """
    # Y = X * H + W => H_LS = X^(-1) * Y
    h_ls_pilot = rx_pilot_signals / pilot_symbols
    return h_ls_pilot


def ls_estimation_full(rx_all_signals, tx_all_symbols):
    """
    全导频LS信道估计（所有子载波都是导频）
    
    参数:
        rx_all_signals: 接收信号 (N,)
        tx_all_symbols: 发送信号 (N,)
    
    返回:
        h_ls_full: 全部子载波的LS估计
    """
    return rx_all_signals / tx_all_symbols


print("LS估计函数定义完成")

### 7.4 MMSE估计实现

In [ ]:
def mmse_estimation(h_ls_pilot, pilot_indices, noise_variance, snr_linear):
    """
    MMSE信道估计
    
    参数:
        h_ls_pilot: 导频位置的LS估计
        pilot_indices: 导频位置索引
        noise_variance: 噪声方差
        snr_linear: 线性信噪比
    
    返回:
        h_mmse_pilot: 导频位置的MMSE估计
    """
    # MMSE权重：w = R_HH / (R_HH + sigma^2)
    # 假设信道功率为1
    channel_power = 1.0
    
    # 计算MMSE权重
    w = channel_power / (channel_power + noise_variance)
    
    # MMSE估计
    h_mmse_pilot = w * h_ls_pilot
    
    return h_mmse_pilot


def mmse_estimation_sparse(h_ls_pilot, pilot_indices, noise_variance, snr_linear, 
                           channel_covariance=None):
    """
    MMSE估计（考虑信道相关性）
    
    对于梳状导频，需要利用频域相关性进行插值
    """
    # 简化版：对每个导频独立应用MMSE
    channel_power = 1.0
    
    # SNR自适应的MMSE权重
    w = channel_power / (channel_power + noise_variance)
    
    h_mmse_pilot = w * h_ls_pilot
    
    return h_mmse_pilot


print("MMSE估计函数定义完成")

### 7.5 频域插值实现

In [ ]:
def interpolate_channel(h_pilot, pilot_indices, N, method='linear'):
    """
    频域插值：从导频位置插值到所有子载波
    
    参数:
        h_pilot: 导频位置的信道估计
        pilot_indices: 导频位置索引
        N: 总子载波数
        method: 插值方法 ('linear', 'spline', 'dft')
    
    返回:
        h_interp: 所有子载波的信道估计
    """
    h_interp = np.zeros(N, dtype=complex)
    
    if method == 'linear':
        # 线性插值
        h_interp[pilot_indices] = h_pilot
        
        # 插值非导频位置
        for i in range(len(pilot_indices) - 1):
            start_idx = pilot_indices[i]
            end_idx = pilot_indices[i + 1]
            
            if end_idx - start_idx > 1:
                # 线性插值
                h_start = h_pilot[i]
                h_end = h_pilot[i + 1]
                for j in range(start_idx, end_idx + 1):
                    alpha = (j - start_idx) / (end_idx - start_idx)
                    h_interp[j] = (1 - alpha) * h_start + alpha * h_end
                    
    elif method == 'dft':
        # DFT插值：利用信道时域稀疏性
        # 步骤1：导频位置设值，其他位置为零
        h_partial = np.zeros(N, dtype=complex)
        h_partial[pilot_indices] = h_pilot
        
        # 步骤2：IFFT到时域
        h_time = np.fft.ifft(h_partial, n=N)
        
        # 步骤3：时域补零（扩展有效长度）
        # 假设信道延迟扩展小于导频间隔
        max_delay = pilot_interval  # 补零长度
        h_time_padded = np.zeros(N, dtype=complex)
        h_time_padded[:max_delay] = h_time[:max_delay]
        
        # 步骤4：FFT回频域
        h_interp = np.fft.fft(h_time_padded, n=N)
        
    else:
        raise ValueError(f"未知插值方法: {method}")
    
    return h_interp


print("频域插值函数定义完成")

### 7.6 时域平滑实现

In [ ]:
def temporal_smoothing(h_current, h_previous, alpha):
    """
    时域平滑
    
    参数:
        h_current: 当前OFDM符号的信道估计
        h_previous: 前一个OFDM符号的信道估计
        alpha: 平滑系数 (0 < alpha <= 1)
    
    返回:
        h_smoothed: 平滑后的信道估计
    """
    return alpha * h_current + (1 - alpha) * h_previous


def adaptive_smoothing(snr_linear):
    """
    自适应平滑系数
    
    参数:
        snr_linear: 线性信噪比
    
    返回:
        alpha: 平滑系数
    """
    # 低SNR时增加平滑，高SNR时减少平滑
    alpha = 1 / (1 + 1 / snr_linear)
    return alpha


print("时域平滑函数定义完成")

### 7.7 SVD-MMSE实现

In [ ]:
def svd_mmse_estimation(h_ls, noise_variance, rank=None):
    """
    SVD-MMSE信道估计
    
    参数:
        h_ls: LS信道估计 (N,)
        noise_variance: 噪声方差
        rank: 保留的秩（默认为信道实际路径数）
    
    返回:
        h_svd_mmse: SVD-MMSE估计
    """
    N = len(h_ls)
    
    # 假设信道自相关矩阵（简化：使用理想相关矩阵）
    # 实际中需要从训练序列估计
    # 这里使用指数衰减模型
    R_HH = np.zeros((N, N), dtype=complex)
    for i in range(N):
        for j in range(N):
            delay = abs(i - j)
            # 指数衰减相关
            R_HH[i, j] = np.exp(-delay / (N / 4)) * np.exp(1j * 0.1 * (i - j))
    
    # SVD分解
    U, s, Vh = linalg.svd(R_HH)
    
    # 确定秩
    if rank is None:
        # 使用能量阈值确定秩
        total_energy = np.sum(s**2)
        cumsum = np.cumsum(s**2)
        rank = np.searchsorted(cumsum, 0.99 * total_energy) + 1
    
    # 截断SVD
    U_r = U[:, :rank]
    s_r = s[:rank]
    Vh_r = Vh[:rank, :]
    
    # SVD-MMSE估计
    # R_HH = U * S * V^H, MMSE = U * S/(S+sigma^2) * V^H * H_LS
    s_reg = s_r / (s_r + noise_variance)
    h_svd_mmse = U_r @ np.diag(s_reg) @ Vh_r @ h_ls
    
    return h_svd_mmse


print("SVD-MMSE估计函数定义完成")

### 7.8 完整估计流程演示

In [ ]:
def simulate_channel_estimation(snr_db, method='ls'):
    """
    模拟完整信道估计流程
    
    参数:
        snr_db: 信噪比（dB）
        method: 估计方法 ('ls', 'mmse', 'svd-mmse')
    
    返回:
        mse: 均方误差
        h_est: 估计的信道频域响应
        h_true: 真实信道频域响应
    """
    # 生成信道
    h_time, h_freq, snr_lin, noise_var = generate_channel(L, channel_delay_spread, snr_db)
    
    # 生成导频符号
    pilot_symbols = generate_pilot_symbols(pilot_indices)
    
    # 计算导频位置的接收信号
    rx_pilots = pilot_symbols * h_freq[pilot_indices]
    
    # 添加噪声
    noise = np.sqrt(noise_var / 2) * (np.random.randn(num_pilots) + 
                                      1j * np.random.randn(num_pilots))
    rx_pilots_noisy = rx_pilots + noise
    
    # LS估计
    h_ls_pilot = ls_estimation(rx_pilots_noisy, pilot_symbols)
    
    if method == 'ls':
        # 直接插值
        h_est = interpolate_channel(h_ls_pilot, pilot_indices, N, method='linear')
        
    elif method == 'mmse':
        # MMSE估计
        h_mmse_pilot = mmse_estimation(h_ls_pilot, pilot_indices, noise_var, snr_lin)
        h_est = interpolate_channel(h_mmse_pilot, pilot_indices, N, method='linear')
        
    elif method == 'svd-mmse':
        # 先LS估计所有子载波
        h_ls_full = interpolate_channel(h_ls_pilot, pilot_indices, N, method='dft')
        # SVD-MMSE
        h_est = svd_mmse_estimation(h_ls_full, noise_var, rank=L)
        
    else:
        raise ValueError(f"未知估计方法: {method}")
    
    # 计算MSE
    mse = np.mean(np.abs(h_est - h_freq)**2)
    
    return mse, h_est, h_freq


# 测试单次估计
mse_ls, h_est_ls, h_true = simulate_channel_estimation(snr_db=20, method='ls')
mse_mmse, h_est_mmse, _ = simulate_channel_estimation(snr_db=20, method='mmse')
mse_svd, h_est_svd, _ = simulate_channel_estimation(snr_db=20, method='svd-mmse')

print(f"SNR = 20 dB 时的估计MSE：")
print(f"  LS估计:     MSE = {mse_ls:.6f}")
print(f"  MMSE估计:   MSE = {mse_mmse:.6f}")
print(f"  SVD-MMSE:   MSE = {mse_svd:.6f}")

## 8. 不同SNR下的性能比较 (Performance vs SNR)

下面我们比较LS估计、MMSE估计和SVD-MMSE在不同SNR下的性能。

In [ ]:
# SNR范围
snr_range = np.arange(0, 31, 5)  # 0 dB 到 30 dB

# 蒙特卡洛仿真参数
num_trials = 100  # 每次SNR重复次数

# 存储MSE
mse_ls_all = []
mse_mmse_all = []
mse_svd_all = []

print("开始蒙特卡洛仿真...")
print(f"SNR范围: {list(snr_range)} dB")
print(f"每次SNR仿真次数: {num_trials}")
print()

for snr_db in snr_range:
    mse_ls_trials = []
    mse_mmse_trials = []
    mse_svd_trials = []
    
    for trial in range(num_trials):
        # LS估计
        mse_ls, _, _ = simulate_channel_estimation(snr_db, method='ls')
        mse_ls_trials.append(mse_ls)
        
        # MMSE估计
        mse_mmse, _, _ = simulate_channel_estimation(snr_db, method='mmse')
        mse_mmse_trials.append(mse_mmse)
        
        # SVD-MMSE估计
        mse_svd, _, _ = simulate_channel_estimation(snr_db, method='svd-mmse')
        mse_svd_trials.append(mse_svd)
    
    mse_ls_all.append(np.mean(mse_ls_trials))
    mse_mmse_all.append(np.mean(mse_mmse_trials))
    mse_svd_all.append(np.mean(mse_svd_trials))
    
    print(f"SNR = {snr_db:2d} dB | LS: {np.mean(mse_ls_trials):.6f} | MMSE: {np.mean(mse_mmse_trials):.6f} | SVD-MMSE: {np.mean(mse_svd_trials):.6f}")

print()
print("仿真完成!")

### 8.1 MSE vs SNR 性能曲线

In [ ]:
# 绘制MSE vs SNR曲线
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 左图：线性坐标
ax1 = axes[0]
ax1.plot(snr_range, mse_ls_all, 'b-o', linewidth=2, markersize=8, label='LS估计')
ax1.plot(snr_range, mse_mmse_all, 'r-s', linewidth=2, markersize=8, label='MMSE估计')
ax1.plot(snr_range, mse_svd_all, 'g-^', linewidth=2, markersize=8, label='SVD-MMSE')
ax1.set_xlabel('信噪比 (dB)', fontsize=12)
ax1.set_ylabel('均方误差 (MSE)', fontsize=12)
ax1.set_title('信道估计MSE vs SNR（线性坐标）', fontsize=13)
ax1.legend(fontsize=11)
ax1.grid(True, alpha=0.3)
ax1.set_yscale('log')

# 右图：理论对比
ax2 = axes[1]
snr_linear = 10**(snr_range / 10)
noise_floor = 1 / snr_linear  # LS的理论MSE下限

ax2.semilogy(snr_range, mse_ls_all, 'b-o', linewidth=2, markersize=8, label='LS估计')
ax2.semilogy(snr_range, mse_mmse_all, 'r-s', linewidth=2, markersize=8, label='MMSE估计')
ax2.semilogy(snr_range, mse_svd_all, 'g-^', linewidth=2, markersize=8, label='SVD-MMSE')
ax2.semilogy(snr_range, noise_floor, 'k--', linewidth=2, label='LS理论下限')
ax2.set_xlabel('信噪比 (dB)', fontsize=12)
ax2.set_ylabel('均方误差 (MSE)', fontsize=12)
ax2.set_title('信道估计MSE vs SNR（对数坐标）', fontsize=13)
ax2.legend(fontsize=11)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("观察：")
print("1. LS估计的MSE与1/SNR成正比（噪声放大效应）")
print("2. MMSE和SVD-MMSE在低SNR时显著优于LS估计")
print("3. 高SNR时，三种方法趋于一致（MMSE退化为LS）")
print("4. SVD-MMSE接近MMSE性能，但复杂度更低")

### 8.2 星座图对比

In [ ]:
# 生成测试数据（使用不同估计方法均衡后的接收星座）
def compare_constellations(snr_db):
    """比较不同估计方法的均衡后星座图"""
    
    # 生成信道和导频
    h_time, h_freq, snr_lin, noise_var = generate_channel(L, channel_delay_spread, snr_db)
    pilot_symbols = generate_pilot_symbols(pilot_indices)
    
    # LS估计 + 均衡
    mse_ls, h_est_ls, _ = simulate_channel_estimation(snr_db, method='ls')
    
    # MMSE估计 + 均衡
    mse_mmse, h_est_mmse, _ = simulate_channel_estimation(snr_db, method='mmse')
    
    # SVD-MMSE估计 + 均衡
    mse_svd, h_est_svd, _ = simulate_channel_estimation(snr_db, method='svd-mmse')
    
    # 生成测试QAM符号
    qpsk = np.array([1+0j, 0+1j, -1+0j, 0-1j]) / np.sqrt(2)
    tx_symbol = qpsk[0]  # 使用一个QAM符号测试
    
    # 接收信号（加噪声）
    rx_symbol_ls = tx_symbol * h_freq + np.sqrt(noise_var / 2) * (np.random.randn() + 1j * np.random.randn())
    rx_symbol_mmse = rx_symbol_ls.copy()
    rx_symbol_svd = rx_symbol_ls.copy()
    
    # 均衡（除以估计的信道）
    rx_equalized_ls = rx_symbol_ls / h_est_ls
    rx_equalized_mmse = rx_symbol_mmse / h_est_mmse
    rx_equalized_svd = rx_symbol_svd / h_est_svd
    
    return {
        'ls': rx_equalized_ls,
        'mmse': rx_equalized_mmse,
        'svd': rx_equalized_svd,
        'true': tx_symbol
    }

# 低SNR和高SNR的星座图对比
fig, axes = plt.subplots(2, 3, figsize=(15, 10))

# 低SNR (5 dB)
snr_low = 5
constellations_low = compare_constellations(snr_low)

# 高SNR (20 dB)
snr_high = 20
constellations_high = compare_constellations(snr_high)

# 绘制星座图
methods = ['ls', 'mmse', 'svd']
method_names = ['LS估计', 'MMSE估计', 'SVD-MMSE']

for i, (method, name) in enumerate(zip(methods, method_names)):
    # 低SNR
    ax = axes[0, i]
    rx_eq = constellations_low[method]
    ax.scatter(rx_eq.real, rx_eq.imag, c='red', s=100, alpha=0.7, edgecolors='black')
    ax.scatter(constellations_low['true'].real, constellations_low['true'].imag, 
               c='blue', s=200, marker='x', linewidths=3, label='发送')
    ax.axhline(y=0, color='k', linewidth=0.5)
    ax.axvline(x=0, color='k', linewidth=0.5)
    ax.set_xlabel('I', fontsize=11)
    ax.set_ylabel('Q', fontsize=11)
    ax.set_title(f'{name} (SNR={snr_low}dB)', fontsize=12)
    ax.grid(True, alpha=0.3)
    ax.set_xlim(-2, 2)
    ax.set_ylim(-2, 2)
    ax.set_aspect('equal')
    if i == 0:
        ax.legend()
    
    # 高SNR
    ax = axes[1, i]
    rx_eq = constellations_high[method]
    ax.scatter(rx_eq.real, rx_eq.imag, c='red', s=100, alpha=0.7, edgecolors='black')
    ax.scatter(constellations_high['true'].real, constellations_high['true'].imag, 
               c='blue', s=200, marker='x', linewidths=3, label='发送')
    ax.axhline(y=0, color='k', linewidth=0.5)
    ax.axvline(x=0, color='k', linewidth=0.5)
    ax.set_xlabel('I', fontsize=11)
    ax.set_ylabel('Q', fontsize=11)
    ax.set_title(f'{name} (SNR={snr_high}dB)', fontsize=12)
    ax.grid(True, alpha=0.3)
    ax.set_xlim(-2, 2)
    ax.set_ylim(-2, 2)
    ax.set_aspect('equal')
    if i == 0:
        ax.legend()

plt.tight_layout()
plt.show()

print("观察：")
print("1. 低SNR时，MMSE和SVD-MMSE的星座点更集中，误差更小")
print("2. 高SNR时，所有方法的星座图都接近理想位置")
print("3. LS估计在低SNR时噪声放大严重，星座点发散")

### 8.3 估计器方差对比

In [ ]:
# 比较不同方法的估计方差
def estimate_variance(snr_db, num_trials=200):
    """估计不同方法的估计方差"""
    
    variances = {'ls': [], 'mmse': [], 'svd': []}
    true_channels = []
    
    for _ in range(num_trials):
        # 生成信道（固定参数，不同随机种子）
        np.random.seed(np.random.randint(0, 10000))
        h_time, h_freq, snr_lin, noise_var = generate_channel(L, channel_delay_spread, snr_db)
        true_channels.append(h_freq.copy())
        
        # LS估计
        mse_ls, h_est_ls, _ = simulate_channel_estimation(snr_db, method='ls')
        variances['ls'].append(h_est_ls.copy())
        
        # MMSE估计
        mse_mmse, h_est_mmse, _ = simulate_channel_estimation(snr_db, method='mmse')
        variances['mmse'].append(h_est_mmse.copy())
        
        # SVD-MMSE估计
        mse_svd, h_est_svd, _ = simulate_channel_estimation(snr_db, method='svd-mmse')
        variances['svd'].append(h_est_svd.copy())
    
    return true_channels, variances


# 计算各子载波的估计方差
snr_test = 10
true_chs, est_chs = estimate_variance(snr_test, num_trials=100)

# 计算每个子载波的方差
var_ls = np.var(np.array(est_chs['ls']) - np.array(true_chs), axis=0)
var_mmse = np.var(np.array(est_chs['mmse']) - np.array(true_chs), axis=0)
var_svd = np.var(np.array(est_chs['svd']) - np.array(true_chs), axis=0)

# 绘制方差分布
fig, ax = plt.subplots(figsize=(12, 5))

subcarrier_idx = np.arange(N)
ax.plot(subcarrier_idx, var_ls, 'b-', alpha=0.7, linewidth=2, label='LS估计方差')
ax.plot(subcarrier_idx, var_mmse, 'r-', alpha=0.7, linewidth=2, label='MMSE估计方差')
ax.plot(subcarrier_idx, var_svd, 'g-', alpha=0.7, linewidth=2, label='SVD-MMSE方差')

ax.set_xlabel('子载波索引', fontsize=12)
ax.set_ylabel('估计方差', fontsize=12)
ax.set_title(f'各子载波估计方差分布 (SNR={snr_test}dB)', fontsize=13)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("观察：")
print("1. LS估计方差在所有子载波上基本恒定（由噪声决定）")
print("2. MMSE和SVD-MMSE在边缘子载波处方差更低（利用了频域相关性）")
print("3. 导频位置处的方差略低（导频处直接估计，无插值误差）")

## 9. 时域平滑性能分析 (Temporal Smoothing Analysis)

下面我们验证时域平滑对估计性能的提升。

In [ ]:
def simulate_with_smoothing(snr_db, alpha):
    """
    带时域平滑的信道估计
    """
    mse_results = []
    h_prev = None
    
    for _ in range(10):  # 10个连续符号
        mse, h_est, h_true = simulate_channel_estimation(snr_db, method='mmse')
        
        if h_prev is not None:
            h_est_smoothed = temporal_smoothing(h_est, h_prev, alpha)
            mse_smoothed = np.mean(np.abs(h_est_smoothed - h_true)**2)
            mse = mse_smoothed
        
        h_prev = h_est.copy()
        mse_results.append(mse)
    
    return np.mean(mse_results)


# 测试不同平滑系数
alpha_values = [0.1, 0.3, 0.5, 0.7, 1.0]
snr_test = 10
snr_test_linear = 10**(snr_test/10)

mse_smoothed = []
for alpha in alpha_values:
    mse = simulate_with_smoothing(snr_test, alpha)
    mse_smoothed.append(mse)
    
print(f"不同平滑系数的MSE (SNR={snr_test}dB)：")
for alpha, mse in zip(alpha_values, mse_smoothed):
    print(f"  alpha = {alpha:.1f}: MSE = {mse:.6f}")

# 绘制平滑效果
fig, ax = plt.subplots(figsize=(10, 5))

ax.bar(range(len(alpha_values)), mse_smoothed, width=0.5, 
       color='steelblue', alpha=0.7, edgecolor='black')
ax.set_xticks(range(len(alpha_values)))
ax.set_xticklabels([f'{a:.1f}' for a in alpha_values])
ax.set_xlabel('平滑系数 α', fontsize=12)
ax.set_ylabel('均方误差 (MSE)', fontsize=12)
ax.set_title(f'时域平滑性能 (SNR={snr_test}dB)', fontsize=13)
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

print("观察：")
print("1. alpha=1.0时，无平滑，等于MMSE估计")
print("2. 适当的平滑（alpha~0.3-0.5）可以降低MSE")
print("3. 过度平滑（alpha过小）会导致跟踪误差增加")

## 10. 方法比较总结 (Method Comparison Summary)

### 10.1 性能对比

In [ ]:
# 绘制综合性能对比表
fig, ax = plt.subplots(figsize=(10, 6))
ax.axis('off')

# 表格数据
methods = ['LS', 'MMSE', 'SVD-MMSE', '时域平滑LS']
complexity = ['O(N)', 'O(N²)或O(N³)', 'O(rN²)', 'O(N)']
memory = ['O(N)', 'O(N²)', 'O(rN)', 'O(N)']
performance_low_snr = ['差', '最优', '接近最优', '中等']
performance_high_snr = ['最优', '最优', '接近最优', '最优']
prior_knowledge = ['无', '信道统计', '信道统计', '时间相关性']

table_data = [
    ['计算复杂度', complexity[0], complexity[1], complexity[2], complexity[3]],
    ['存储需求', memory[0], memory[1], memory[2], memory[3]],
    ['低SNR性能', performance_low_snr[0], performance_low_snr[1], performance_low_snr[2], performance_low_snr[3]],
    ['高SNR性能', performance_high_snr[0], performance_high_snr[1], performance_high_snr[2], performance_high_snr[3]],
    ['所需先验知识', prior_knowledge[0], prior_knowledge[1], prior_knowledge[2], prior_knowledge[3]]
]

table = ax.table(
    cellText=table_data,
    rowLabels=['计算复杂度', '存储需求', '低SNR性能', '高SNR性能', '所需先验知识'],
    colLabels=['', 'LS', 'MMSE', 'SVD-MMSE', '时域平滑LS'],
    loc='center',
    cellLoc='center'
)
table.auto_set_font_size(False)
table.set_fontsize(11)
table.scale(1.2, 1.8)

# 设置表头样式
for i in range(len(methods) + 1):
    table[(0, i)].set_facecolor('#4472C4')
    table[(0, i)].set_text_props(color='white', fontweight='bold')

ax.set_title('信道估计方法综合对比', fontsize=14, fontweight='bold', pad=20)

plt.tight_layout()
plt.show()

### 10.2 应用场景建议

In [ ]:
# 绘制应用场景建议
fig, ax = plt.subplots(figsize=(12, 4))
ax.axis('off')

scenarios = [
    ['场景', '推荐方法', '原因'],
    ['高SNR、计算资源充足', 'MMSE', 'MMSE在高SNR下性能最优'],
    ['低SNR、计算资源充足', 'MMSE', 'MMSE有效抑制噪声放大'],
    ['高SNR、计算资源受限', 'LS', 'LS简单高效，高SNR下性能损失小'],
    ['低SNR、计算资源受限', 'SVD-MMSE', '复杂度低，性能接近MMSE'],
    ['时变信道、需跟踪', '时域平滑+MMSE', '利用时间相关性跟踪信道变化'],
    ['稀疏信道', '压缩感知MMSE', '利用信道稀疏性提升精度'],
]

table = ax.table(
    cellText=scenarios[1:],
    colLabels=scenarios[0],
    loc='center',
    cellLoc='center'
)
table.auto_set_font_size(False)
table.set_fontsize(11)
table.scale(1.2, 1.8)

# 表头样式
for i in range(3):
    table[(0, i)].set_facecolor('#4472C4')
    table[(0, i)].set_text_props(color='white', fontweight='bold')

ax.set_title('信道估计方法应用场景建议', fontsize=14, fontweight='bold', pad=20)

plt.tight_layout()
plt.show()

## 11. 思考题 (Review Questions)

1. **LS vs MMSE**：解释为什么在低SNR条件下，MMSE估计的性能优于LS估计。LS估计的"噪声放大"效应在数学上是如何体现的？

2. **信道先验的重要性**：MMSE估计需要知道信道的统计特性（如自相关矩阵）。如果这些先验知识不准确（例如模型与实际信道失配），MMSE估计的性能会如何变化？这叫什么问题？

3. **SVD-MMSE的物理意义**：SVD-MMSE通过截断SVD保留主奇异值来降低复杂度。请解释：
   - 为什么信道自相关矩阵的前$r$个奇异值包含大部分能量？
   - 截断SVD在物理上相当于对信道做了什么假设？
   - 如果保留的秩$r$太大或太小，会发生什么问题？

4. **贝叶斯框架**：在贝叶斯估计框架下，MAP估计和MMSE估计分别在什么条件下等价？当信道服从均匀分布而非高斯分布时，MAP估计还等于MMSE吗？为什么？

5. **时域平滑的极限**：假设信道是静态的（不随时间变化），时域平滑的最佳$\alpha$值是多少？如果信道快速变化，最佳$\alpha$会如何变化？这说明时域平滑适用于什么类型的信道？

6. **梳状导频插值误差**：梳状导频下，LS估计后进行线性插值会在导频之间引入插值误差。请分析：
   - 插值误差与导频间隔有什么关系？
   - 如何利用信道时域稀疏性来减小插值误差？
   - DFT插值方法为什么能利用时域稀疏性？

7. **复杂度与性能权衡**：对于$N=1024$子载波的OFDM系统，请比较以下三种方法的计算复杂度和存储需求：
   - 精确MMSE
   - SVD-MMSE（$r=8$）
   - 对角化MMSE（假设信道自相关矩阵是对角的）

8. **OTFS信道估计的特殊性**：与OFDM不同，OTFS在延迟-多普勒域进行信道估计。请思考：
   - 在OTFS中，LS估计和MMSE估计的形式会有什么变化？
   - OTFS信道在延迟-多普勒域的稀疏性如何帮助提升估计性能？
   - 如果信道是多普勒扩展的（高速移动场景），在OFDM和OTFS中分别如何处理？

---

## 总结 (Summary)

本notebook介绍了信道估计的两种基本方法及其低复杂度变体：

- **LS估计**：简单高效，但存在噪声放大问题，适用于高SNR场景
- **MMSE估计**：利用信道统计特性，在低SNR下显著优于LS，但计算复杂度高
- **SVD-MMSE**：通过截断SVD降低复杂度，性能接近MMSE
- **时域平滑**：利用时间相关性提升估计精度，适用于时变信道

关键洞察：**估计器的性能取决于是否有效利用了信道的结构特性**（频域相关性、时间相关性、稀疏性）。越多的先验知识通常带来越好的估计性能，但也需要更高的计算复杂度。